# StyleGAN2-ADA on Kaggle — smoke run

Trains one class for 20 kimg to confirm the pipeline works and measure speed.

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**, and add the
Kaggle Dataset built from `data/stylegan.zip` as an Input.

Trains on **one** GPU on purpose. `train.py` runs single-GPU in-process, while
multi-GPU goes through `multiprocessing.spawn` + DDP and hides real tracebacks
behind `ProcessRaisedException`. Half the speed, far less to go wrong.

Takes about 15 minutes. Only one thing genuinely needs a human eye: the sample
grid in step 3 must look like **dogs**, which proves the pretrained weights
loaded.

### Upstream bugs patched in step 1

NVlabs targets torch 1.7–1.10; Kaggle ships 2.x.

- **`custom_ops.py`** discards `cpp_extension.load()`'s return value and then
  re-imports the module by name. Modern PyTorch does not leave it importable
  that way, so the CUDA kernels report *"Failed!"* after compiling fine.
- **`misc.py`** calls `Sampler.__init__(dataset)`, dropped in torch 2.x.
- **`grid_sample_gradfix.py`** supplies the *second* derivative of
  `grid_sample`, which torch still lacks. R1 differentiates through the ADA
  augment pipeline twice, so with this disabled training dies partway through
  the first tick with *"derivative for aten::grid_sampler_2d_backward is not
  implemented"*. Its version gate switches it off on 2.x, and the op it wraps
  grew a 7th argument since. Both are fixed.

`conv2d_gradfix` is deliberately **left disabled** — the op it wraps was deleted
from PyTorch outright, and plain `F.conv2d` handles double backward fine. Its
`Falling back to torch.nn.functional.conv2d()` warning is correct. Leave it.

## 1. Setup

In [ ]:
import os, sys, json, time, pathlib, subprocess, shutil
import torch

REPO = "/kaggle/working/stylegan2-ada-pytorch"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "-q",
                "https://github.com/NVlabs/stylegan2-ada-pytorch.git", REPO], check=True)
sys.path.insert(0, REPO)

def patch(rel, old, new):
    f = pathlib.Path(REPO) / rel
    s = f.read_text()
    assert old in s, f"patch target not found in {rel}"   # never fail silently
    f.write_text(s.replace(old, new))

# custom_ops.py drops load()'s return value then re-imports by name, which
# modern PyTorch does not support -- the kernels "fail" after building fine.
patch("torch_utils/custom_ops.py",
      "torch.utils.cpp_extension.load(name=module_name",
      "module = torch.utils.cpp_extension.load(name=module_name")
patch("torch_utils/custom_ops.py",
      "        module = importlib.import_module(module_name)\n", "")

# PyTorch 2.x dropped Sampler.__init__(data_source), so this hits object.__init__.
patch("torch_utils/misc.py", "super().__init__(dataset)", "super().__init__()")

# grid_sample_gradfix supplies the SECOND derivative of grid_sample, which torch
# still does not have ("derivative for aten::grid_sampler_2d_backward is not
# implemented"). The R1 penalty differentiates through the ADA augment pipeline
# twice, so without this training dies at loss.py:131. Its version gate disables
# it on torch 2.x, and the op it wraps grew a 7th argument. Both need fixing.
GSG = "torch_utils/ops/grid_sample_gradfix.py"
patch(GSG, "any(torch.__version__.startswith(x) for x in ['1.7.', '1.8.', '1.9'])",
      "True")
patch(GSG,
      "op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')\n"
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False)",
      "op = torch.ops.aten.grid_sampler_2d_backward\n"
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False, [True, True])")

# --gpus=1 deliberately, even on T4 x2. train.py runs single-GPU IN-PROCESS;
# --gpus=2 goes through multiprocessing.spawn + DistributedDataParallel, which
# wraps every error in ProcessRaisedException and hides the real traceback.
# Half the speed, a fraction of the failure surface. Revisit once it works.
NUM_GPUS = 1
print("torch:", torch.__version__, "|", torch.cuda.get_device_name(0),
      "| visible gpus:", torch.cuda.device_count(), "-> using", NUM_GPUS)

## 2. Compile the CUDA kernels, and check the R1 path

First use of `bias_act` / `upfirdn2d` triggers an `nvcc` build — 2–5 minutes,
silent while it works. Success here is worth waiting for: without it these fall
back to slow reference code and training takes several times longer.

The second check takes a fraction of a second and is worth far more than that:
it runs the exact double-backward the R1 penalty needs. If the
`grid_sample_gradfix` patch did not take, this fails here rather than ten
minutes into training.

In [ ]:
from torch_utils.ops import bias_act, upfirdn2d, grid_sample_gradfix

t0 = time.time()
bias_act.bias_act(torch.randn(4, 8, device="cuda"),
                  torch.randn(8, device="cuda"), impl="cuda")
upfirdn2d.upfirdn2d(torch.randn(2, 3, 16, 16, device="cuda"),
                    torch.ones(4, 4, device="cuda") / 16, impl="cuda")
torch.cuda.synchronize()

ok = bias_act._init() and upfirdn2d._init()
print(f"built in {time.time()-t0:.0f}s")
print("CUDA kernels:", "compiled" if ok else "FAILED - training will be slow")

# The R1 penalty differentiates through the augment pipeline twice. This is
# that operation, in miniature.
grid_sample_gradfix.enabled = True
img = torch.randn(2, 3, 16, 16, device="cuda", requires_grad=True)
grid = torch.rand(2, 16, 16, 2, device="cuda") * 2 - 1
d = torch.nn.Conv2d(3, 1, 3, padding=1).cuda()
g = torch.autograd.grad(d(grid_sample_gradfix.grid_sample(img, grid)).sum(),
                        img, create_graph=True)[0]
g.square().sum().backward()
print("R1 double backward: OK")

## 3. Load the pretrained model

LSUN Dog at 256px — full-body animals, the closest public checkpoint to
full-body creatures.

**The grid below must look like dogs.** If it is noise, the weights did not
load and any training from here would silently start from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import legacy, dnnlib

URL = ("https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/"
       "transfer-learning-source-nets/lsundog-res256-paper256-kimg100000-noaug.pkl")
PKL = "/kaggle/working/lsundog-res256.pkl"
if not os.path.exists(PKL):
    subprocess.run(["wget", "-q", "-O", PKL, URL], check=True)

with dnnlib.util.open_url(PKL) as f:
    G = legacy.load_network_pkl(f)["G_ema"].to("cuda")

z = torch.from_numpy(np.random.RandomState(0).randn(8, G.z_dim)).to("cuda")
with torch.no_grad():
    img = G(z, None, truncation_psi=0.7, noise_mode="const")
img = (img.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()

fig, axes = plt.subplots(1, 8, figsize=(16, 2.2))
for ax, im in zip(axes, img):
    ax.imshow(im); ax.axis("off")
plt.suptitle("must look like dogs", fontsize=12)
plt.show()
print("resolution:", G.img_resolution)

## 4. Build the dataset

Uses NVlabs' `dataset_tool.py`, which produces the exact zip layout
`training/dataset.py` expects.

In [ ]:
DATA = next(p.parent for p in pathlib.Path("/kaggle/input").glob("**/summary.json"))
counts = json.loads((DATA / "summary.json").read_text())["classes"]
print("classes:", {k: v["count"] for k, v in counts.items()})

CLASS = "arthropod"
ZIP = f"/kaggle/working/{CLASS}.zip"
subprocess.run([sys.executable, f"{REPO}/dataset_tool.py",
                f"--source={DATA / CLASS}", f"--dest={ZIP}"], check=True)

from training.dataset import ImageFolderDataset
ds = ImageFolderDataset(path=ZIP, use_labels=False, max_size=None, xflip=False)
print(f"{CLASS}: {len(ds)} images, {ds.image_shape}")

## 5. Train — 20 kimg

`--cfg=paper256` must match the pretrained model's config or `--resume` fails
on mismatched layer shapes. Expect 8–10 minutes.

`conv2d_gradfix` warns on **every** convolution — thousands of identical lines
that bury the progress ticks and would bury a real traceback too. It is
expected, so it is filtered out here and the rest streams live.

The last line on success is `Exiting...`, printed by `training_loop.py` only
after the run completes.

In [ ]:
OUT = "/kaggle/working/smoke"
cmd = [sys.executable, f"{REPO}/train.py",
       f"--outdir={OUT}", f"--data={ZIP}", f"--gpus={NUM_GPUS}",
       "--cfg=paper256", "--mirror=1", "--aug=ada", "--target=0.6",
       f"--resume={PKL}", "--snap=1", "--metrics=none", "--kimg=20"]

t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    if "conv2d_gradfix" not in line:      # expected, and thousands of them
        print(line, end="")
p.wait()

elapsed = time.time() - t0
if p.returncode != 0:
    raise SystemExit(f"training failed (exit {p.returncode})")
print(f"done in {elapsed/60:.1f} min")

## 6. Speed and results

`sec/kimg` is measured between ticks, not from total time — the first tick
includes kernel compilation and would skew it badly.

In [ ]:
run = sorted(pathlib.Path(OUT).glob("00000-*"))[-1]
ticks = [json.loads(l) for l in (run / "stats.jsonl").read_text().splitlines() if l.strip()]
get = lambda t, k: t.get(k, {}).get("mean", 0)

kimg = get(ticks[-1], "Progress/kimg")
if len(ticks) >= 3:
    sec_per_kimg = ((get(ticks[-1], "Timing/total_sec") - get(ticks[1], "Timing/total_sec"))
                    / (kimg - get(ticks[1], "Progress/kimg")))
else:
    sec_per_kimg = elapsed / kimg

print(f"sec/kimg: {sec_per_kimg:.1f}")
for b in (200, 400, 600):
    print(f"  {b} kimg per class -> {sec_per_kimg*b/3600:.1f} h")
print()
for t in ticks:
    print(f"  kimg {get(t,'Progress/kimg'):5.0f}  G {get(t,'Loss/G/loss'):7.3f}"
          f"  D {get(t,'Loss/D/loss'):7.3f}  ada_p {get(t,'Progress/augment'):.3f}")

snap = sorted(run.glob("network-snapshot-*.pkl"))[-1]
with dnnlib.util.open_url(str(snap)) as f:
    G2 = legacy.load_network_pkl(f)["G_ema"].to("cuda")
with torch.no_grad():
    img2 = G2(z, None, truncation_psi=0.7, noise_mode="const")
print(f"\nchanged from source by {(img2 - G(z, None, truncation_psi=0.7, noise_mode='const')).abs().mean():.4f}")

img2 = (img2.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()
fig, axes = plt.subplots(1, 8, figsize=(16, 2.2))
for ax, im in zip(axes, img2):
    ax.imshow(im); ax.axis("off")
plt.suptitle(f"after {kimg:.0f} kimg - still dog-like, but drifting", fontsize=12)
plt.show()

## Done

Report back: the **sec/kimg** from step 6, and whether step 2 said *compiled* or
*FAILED*. Those two numbers size the real per-class training runs.